In [1]:
%pip install \
    "qiskit==2.2.3" \
    "qiskit-machine-learning==0.9.0" \
    "qiskit-optimization==0.7.0" \
    "qiskit-algorithms==0.4.0" \
    "packaging>=24.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.0/664.0 kB 11.7 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 20.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.7 MB/s  0:00:00
  Created wheel for docplex: filename=docplex-2.32.264-py3-none-any.whl size=703859 sha256=f83cc776bd5f4c507b5d3285564e56b0b1154d8511ff75d842b77e453eb9b112
  Stored in directory: /root/.cache/pip/wheels/ba/11/de/5d0051be010658975fb2155006b4b8a818136d9f69ed20b4ba
Successfully built docplex
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [qiskit-algorithms]


In [6]:
import numpy as np
from importlib.metadata import version
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

print('Qiskit:', version('qiskit'))
data = np.array([0.3, 0.7, 0.5])
theta = np.pi * data  # 세타: 특징을 변환한 RY 각도, 라디안
qc = QuantumCircuit(len(data))
for i, angle in enumerate(theta):
    qc.ry(float(angle), i)
state = Statevector.from_instruction(qc)
measured = np.array([state.probabilities([i])[1] for i in range(len(data))])
expected = np.sin(theta / 2)**2
print('각도:', np.round(theta, 6))
print('각 큐비트의 1 확률:', np.round(measured, 6))
assert np.allclose(measured, expected)

Qiskit: 2.2.3
각도: [0.942478 2.199115 1.570796]
각 큐비트의 1 확률: [0.206107 0.793893 0.5     ]


In [7]:
import numpy as np
from importlib.metadata import version
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

print('Qiskit:', version('qiskit'))
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import PauliEvolutionGate

x = 0.4  # 무차원 시간 매개변수
H = SparsePauliOp.from_list([('ZI', 1.0), ('IZ', 1.0)])
qc = QuantumCircuit(2)
qc.h(range(2))
qc.append(PauliEvolutionGate(H, time=x), [0, 1])
after_evolution = Statevector.from_instruction(qc)
qc.h(range(2))
state = Statevector.from_instruction(qc)
print('시간 발전 후 Z측정 확률:', np.round(after_evolution.probabilities(), 6))
print('H 적용 후 확률:', np.round(state.probabilities(), 6))
p1 = np.array([state.probabilities([i])[1] for i in range(2)])
print('각 큐비트의 1 확률:', np.round(p1, 6))
assert np.allclose(p1, np.sin(x)**2)

Qiskit: 2.2.3
시간 발전 후 Z측정 확률: [0.25 0.25 0.25 0.25]
H 적용 후 확률: [0.719703 0.12865  0.12865  0.022997]
각 큐비트의 1 확률: [0.151647 0.151647]


/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)


시간 발전 직후의 Z측정 확률은 균등합니다. 마지막 H 이후에는 입력 시간에 따라 확률이 달라집니다. 상대 위상을 관측하는 측정 기저가 중요합니다.: